# 05. Análise dos dados brutos da dengue

Estuda a tabela bruta `sinan_dengue` antes da limpeza, para decidir como recortar o Nordeste e o que tratar.
Só consulta o banco: nada é alterado.

**Neste notebook**
1. Código do município de residência
2. Ligação com os municípios do IBGE
3. Residentes do Nordeste
4. Ano da notificação × ano do arquivo
5. Estado de residência × município
6. Notificações duplicadas
7. Classificação final
8. Validação das regras da limpeza, coluna por coluna
9. Resumo

In [1]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
import psycopg
from config import BANCO

con = psycopg.connect(**BANCO, autocommit=True)

def consulta(q):
    cur = con.execute(q)
    return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 1. Código do município de residência
`id_mn_resi` é o código IBGE do município onde a pessoa mora. Quantos dígitos ele tem em cada período?

In [2]:
consulta("""
    select case when ano_arquivo <= 2006 then '2000-2006' else '2007-2026' end as periodo,
           coalesce(length(id_mn_resi)::text, 'vazio') as digitos, count(*) as notificacoes
    from sinan_dengue group by 1, 2 order by 1, 2""")

,periodo,digitos,notificacoes
0,2000-2006,6,19
1,2000-2006,7,2783637
2,2000-2006,vazio,881
3,2007-2026,1,1
4,2007-2026,3,2
5,2007-2026,5,1
6,2007-2026,6,29724685
7,2007-2026,vazio,2776


## 2. Ligação com os municípios do IBGE
Os 6 primeiros dígitos do código devem existir na tabela `municipio` (`cod_mun6`). Para os códigos de 7 dígitos,
o 7º dígito (verificador) deve ser igual ao do IBGE.

Para as próximas consultas, os residentes do Nordeste (código começando de 21 a 29) vão para uma tabela temporária,
que existe só enquanto o notebook está aberto.

In [3]:
con.execute("""
    create temp table ne as
    select ano_arquivo, nu_ano, id_mn_resi, left(id_mn_resi, 6) as cod6, id_municip, sg_uf,
           dt_notific, dt_sin_pri, cs_sexo, nu_idade, nu_idade_n, id_unidade,
           con_classi, classi_fin, ano_nasc,
           coalesce(criterio, con_criter) as criterio, coalesce(evolucao, con_evoluc) as evolucao
    from sinan_dengue where left(id_mn_resi, 2) between '21' and '29'""")

consulta("""
    select
      (select count(*) from ne) as notificacoes_ne,
      (select count(*) from ne join municipio m on m.cod_mun6 = ne.cod6) as batem_com_ibge,
      (select count(*) from ne left join municipio m on m.cod_mun6 = ne.cod6 where m.cod_mun6 is null) as nao_batem,
      (select count(*) from ne join municipio m on m.cod_mun6 = ne.cod6
        where length(id_mn_resi) = 7 and id_mn_resi <> m.cod_mun7) as setimo_digito_diferente,
      (select count(distinct cod6) from ne join municipio m on m.cod_mun6 = ne.cod6) as municipios_com_notificacao""").T.rename(columns={0: "valor"})

,valor
notificacoes_ne,5598577
batem_com_ibge,5598410
nao_batem,167
setimo_digito_diferente,0
municipios_com_notificacao,1794


### 2.1 Códigos sem par no IBGE

In [4]:
consulta("""
    select id_mn_resi as codigo, count(*) as notificacoes, min(ano_arquivo) as de, max(ano_arquivo) as ate
    from ne left join municipio m on m.cod_mun6 = ne.cod6
    where m.cod_mun6 is null group by 1 order by 2 desc""")

,codigo,notificacoes,de,ate
0,2900000,34,2000,2005
1,2400000,27,2000,2005
2,2100000,25,2000,2002
3,2300000,20,2001,2005
4,290000,13,2008,2024
5,2500000,11,2000,2005
6,2600000,10,2000,2002
7,2800000,10,2001,2005
8,2200000,4,2001,2002
9,260000,4,2008,2013


## 3. Residentes do Nordeste

### 3.1 Notificações sem município de residência (Brasil)
Sem o código não dá para saber a região, então essas notificações não entram em nenhum recorte.

In [5]:
consulta("""
    select ano_arquivo as ano, count(*) as sem_municipio
    from sinan_dengue where id_mn_resi is null group by 1 order by 1""").set_index("ano").T

ano,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
sem_municipio,1,34,255,311,67,131,82,200,68,106,...,9,20,52,35,77,193,8,21,9,7


### 3.2 Notificados fora do Nordeste
Residentes do Nordeste notificados em municípios de outras regiões (ex.: atendidos em São Paulo). Entram no recorte
por residência; ficariam de fora num recorte por local de notificação.

In [6]:
consulta("""
    select ano_arquivo as ano, count(*) as residentes_ne,
           count(*) filter (where left(id_municip, 2) not between '21' and '29') as notificados_fora,
           round(100.0 * count(*) filter (where left(id_municip, 2) not between '21' and '29') / count(*), 2) as pct_fora
    from ne group by 1 order by 1""").set_index("ano")

,residentes_ne,notificados_fora,pct_fora
ano,,,
2000,83260,42,0.05
2001,181002,83,0.05
2002,326162,125,0.04
2003,207681,64,0.03
2004,41861,35,0.08
2005,124677,42,0.03
2006,107714,32,0.03
2007,175517,62,0.04
2008,299721,120,0.04


## 4. Ano da notificação × ano do arquivo
O CSV de um ano pode trazer notificações com outro `nu_ano`. Quantas e de quais anos?

In [7]:
consulta("""
    select ano_arquivo as ano, count(*) as notificacoes,
           count(*) filter (where nu_ano <> ano_arquivo::text) as nu_ano_diferente,
           string_agg(distinct nu_ano, ', ') filter (where nu_ano <> ano_arquivo::text) as quais
    from ne group by 1 order by 1""").set_index("ano")

,notificacoes,nu_ano_diferente,quais
ano,,,
2000,83260,1,1999
2001,181002,40,2000
2002,326162,28,2001
2003,207681,1,2002
2004,41861,0,NaN
2005,124677,0,NaN
2006,107714,0,NaN
2007,175517,0,NaN
2008,299721,2,"20¸, 2408"


## 5. Estado de residência × município
`sg_uf` (estado de residência) vem como sigla nos anos antigos e como código nos recentes. Ele bate com os dois
primeiros dígitos do município?

In [8]:
consulta("""
    select case when ano_arquivo <= 2006 then '2000-2006' else '2007-2026' end as periodo,
           count(*) as notificacoes,
           count(*) filter (where ne.sg_uf in (e.sigla, e.uf_cod::text)) as coerentes,
           count(*) filter (where ne.sg_uf is null) as sg_uf_vazio,
           count(*) filter (where ne.sg_uf is not null and ne.sg_uf not in (e.sigla, e.uf_cod::text)) as diferentes
    from ne join estado e on e.uf_cod::text = left(ne.id_mn_resi, 2)
    group by 1 order by 1""")

,periodo,notificacoes,coerentes,sg_uf_vazio,diferentes
0,2000-2006,1072357,1072323,0,34
1,2007-2026,4526220,4526079,85,56


## 6. Possíveis duplicadas

Uma notificação duplicada é **a mesma pessoa registrada duas vezes** (ficha digitada de novo, ou pessoa notificada
outra vez ao voltar ao serviço). Se as duas linhas ficam, a pessoa conta como dois casos e a incidência sobe.

Os dados públicos são **anonimizados** (sem nome, CPF ou cartão do SUS), então não dá para ter certeza. Aqui chamamos
de *possível duplicada* a linha igual a outra em: município e unidade que notificaram, data da notificação, data dos
primeiros sintomas, sexo, idade e município de residência. Podem ser duplicidades ou pessoas diferentes que coincidem
nesses dados (mais provável em cidades grandes, em plena epidemia).

In [9]:
consulta("""
    select ano_arquivo as ano, sum(n) as notificacoes, sum(n - 1) filter (where n > 1) as linhas_repetidas,
           round(100.0 * sum(n - 1) filter (where n > 1) / sum(n), 2) as pct_repetidas
    from (select ano_arquivo, count(*) as n from ne
          group by ano_arquivo, id_municip, id_unidade, dt_notific, dt_sin_pri, cs_sexo,
                   coalesce(nu_idade_n, nu_idade), id_mn_resi) g
    group by 1 order by 1""").set_index("ano")

,notificacoes,linhas_repetidas,pct_repetidas
ano,,,
2000,83260,1168,1.40
2001,181002,3476,1.92
2002,326162,8761,2.69
2003,207681,3516,1.69
2004,41861,288,0.69
2005,124677,3132,2.51
2006,107714,2255,2.09
2007,175517,2717,1.55
2008,299721,6546,2.18


### 6.1 Exemplo
Duas linhas de Assú (RN) em 2024: tudo igual, só a data de digitação muda (24 e 25/01). Muito provavelmente a mesma
ficha lançada duas vezes.

In [10]:
consulta("""
    with grupo as (
        select id_municip, id_unidade, dt_notific, dt_sin_pri, cs_sexo, nu_idade_n, id_mn_resi
        from sinan_dengue_2024 where left(id_mn_resi, 2) = '24'
        group by 1, 2, 3, 4, 5, 6, 7 having count(*) = 2 limit 1)
    select s.id_municip, s.id_unidade, s.dt_notific, s.dt_sin_pri, s.cs_sexo, s.nu_idade_n, s.ano_nasc,
           s.cs_raca, s.cs_escol_n, s.id_mn_resi, s.classi_fin, s.dt_digita
    from sinan_dengue_2024 s
    join grupo using (id_municip, id_unidade, dt_notific, dt_sin_pri, cs_sexo, nu_idade_n, id_mn_resi)""")

,id_municip,id_unidade,dt_notific,dt_sin_pri,cs_sexo,nu_idade_n,ano_nasc,cs_raca,cs_escol_n,id_mn_resi,classi_fin,dt_digita
0,240140,9131051,2024-01-17,2024-01-06,M,4060,1963,4,9,240140,8,2024-01-25
1,240140,9131051,2024-01-17,2024-01-06,M,4060,1963,4,9,240140,8,2024-01-24


### 6.2 O campo de duplicidade do próprio SINAN
O SINAN tem o campo `nduplic_n` para marcar duplicidades identificadas pela vigilância, mas no Nordeste ele está
quase sempre vazio, então não dá para depender dele (o significado dos códigos 0, 1 e 2 precisa ser conferido no
dicionário oficial).

In [11]:
consulta("""
    select coalesce(nduplic_n, 'vazio') as nduplic_n, count(*) as notificacoes
    from sinan_dengue where left(id_mn_resi, 2) between '21' and '29' and ano_arquivo >= 2007
    group by 1 order by 2 desc""")

,nduplic_n,notificacoes
0,vazio,4511357
1,1,9316
2,0,5541
3,2,6


### 6.3 Como tratar
Na limpeza, as possíveis duplicadas **não são apagadas**: ganham a marca `possivel_duplicata`. Assim dá para contar
com e sem elas e mostrar no relatório quanto o resultado muda (esperado: pouco, de 0,7% a 3,3% por ano).

## 7. Classificação final
É o campo que decide se um caso foi confirmado, descartado ou ficou inconclusivo. Até 2006 se chama `con_classi`;
depois, `classi_fin`. Quanto está preenchido e quais códigos aparecem em cada ano?

In [12]:
classif = consulta("""
    select ano_arquivo as ano, coalesce(classi_fin, con_classi, 'vazio') as codigo, count(*) as n
    from ne group by 1, 2""")
tabela = classif.pivot_table(index="ano", columns="codigo", values="n", fill_value=0)
(tabela.div(tabela.sum(axis=1), axis=0) * 100).round(1)

codigo,0,1,10,11,12,2,3,4,5,8,vazio
ano,,,,,,,,,,,
2000,9.8,61.0,0.0,0.0,0.0,0.0,0.1,0.0,15.7,0.4,12.9
2001,10.4,52.6,0.0,0.0,0.0,0.3,0.1,0.0,17.2,1.7,17.6
2002,7.8,55.4,0.0,0.0,0.0,0.4,0.2,0.0,17.9,1.1,17.2
2003,1.6,52.2,0.0,0.0,0.0,0.2,0.3,0.0,27.0,1.3,17.4
2004,0.0,40.2,0.0,0.0,0.0,0.2,0.2,0.0,46.6,0.7,12.1
2005,0.0,48.6,0.0,0.0,0.0,0.5,0.3,0.0,39.0,1.1,10.6
2006,0.0,53.9,0.0,0.0,0.0,0.5,0.5,0.0,35.2,0.7,9.1
2007,0.0,50.6,0.0,0.0,0.0,0.9,0.6,0.0,28.4,16.2,3.2
2008,0.0,48.0,0.0,0.0,0.0,1.1,0.5,0.0,30.2,16.2,4.0


Códigos (a confirmar no dicionário oficial do SINAN): até 2013, 1 a 4 = confirmado (formas da doença);
de 2014 em diante, 10, 11 e 12 = confirmado (dengue, com sinais de alarme, grave). Nos dois períodos, 5 = descartado
e 8 = inconclusivo. A tabela acima está em % das notificações de cada ano.

## 8. Validação das regras da limpeza, coluna por coluna
A tabela tratada (`notificacao_dengue_ne`) vai converter datas, decodificar idade, padronizar sexo e unificar códigos.
Aqui conferimos, nos 5,6 milhões de notificações do Nordeste, se os valores do bruto se comportam como essas regras supõem.

### 8.1 Datas
Formato `AAAA-MM-DD`, faixa de valores e intervalo entre os primeiros sintomas e a notificação.

In [13]:
consulta("""
    select campo, vazias, formato_invalido, menor, maior from (
      select 'dt_notific' as campo, count(*) filter (where dt_notific is null) as vazias,
             count(*) filter (where dt_notific !~ '^\\d{4}-\\d{2}-\\d{2}$') as formato_invalido,
             min(dt_notific) filter (where dt_notific ~ '^\\d{4}-\\d{2}-\\d{2}$') as menor,
             max(dt_notific) filter (where dt_notific ~ '^\\d{4}-\\d{2}-\\d{2}$') as maior from ne
      union all
      select 'dt_sin_pri', count(*) filter (where dt_sin_pri is null),
             count(*) filter (where dt_sin_pri !~ '^\\d{4}-\\d{2}-\\d{2}$'),
             min(dt_sin_pri) filter (where dt_sin_pri ~ '^\\d{4}-\\d{2}-\\d{2}$'),
             max(dt_sin_pri) filter (where dt_sin_pri ~ '^\\d{4}-\\d{2}-\\d{2}$') from ne) t""")

,campo,vazias,formato_invalido,menor,maior
0,dt_notific,0,1,2000-01-01,2026-10-03
1,dt_sin_pri,16,3,0006-02-02,2043-11-05


In [14]:
# dias entre os primeiros sintomas e a notificação
consulta("""
    with d as (select dt_notific::date - dt_sin_pri::date as dias from ne
               where dt_notific ~ '^\\d{4}-\\d{2}-\\d{2}$' and dt_sin_pri ~ '^\\d{4}-\\d{2}-\\d{2}$')
    select count(*) filter (where dias < 0) as sintomas_depois_da_notificacao,
           count(*) filter (where dias between 0 and 30) as ate_30_dias,
           count(*) filter (where dias between 31 and 365) as de_31_a_365_dias,
           count(*) filter (where dias > 365) as mais_de_1_ano,
           percentile_cont(0.5) within group (order by dias) as mediana_dias
    from d""")

,sintomas_depois_da_notificacao,ate_30_dias,de_31_a_365_dias,mais_de_1_ano,mediana_dias
0,13,5426323,159205,13016,3.0


### 8.2 Sexo

In [15]:
consulta("""
    select coalesce(cs_sexo, 'vazio') as sexo, count(*) as notificacoes
    from ne group by 1 order by 2 desc""")

,sexo,notificacoes
0,F,3174965
1,M,2416843
2,I,6648
3,vazio,120
4,O,1


### 8.3 Idade
A idade é um código: o 1º caractere é a unidade e o resto é o valor. Ficha antiga (até 2006): letra (`A` anos,
`M` meses, `D` dias, `I999` ignorado). Ficha nova: dígito (`4` anos, `3` meses, `2` dias, `1` horas).
Em alguns anos o arquivo traz a idade como decimal (`4023.0`).

In [16]:
consulta("""
    select ano_arquivo as ano,
           count(*) filter (where coalesce(nu_idade_n, nu_idade) ~ '^([A-Z]|\\d)\\d{3}$') as formato_normal,
           count(*) filter (where nu_idade_n ~ '^\\d{4}\\.0$') as decimal_ponto_zero,
           count(*) filter (where coalesce(nu_idade_n, nu_idade) is null) as vazia,
           count(*) filter (where coalesce(nu_idade_n, nu_idade) !~ '^([A-Z]|\\d)\\d{3}(\\.0)?$') as sem_unidade_ou_estranha
    from ne group by 1 order by 1""").set_index("ano")

,formato_normal,decimal_ponto_zero,vazia,sem_unidade_ou_estranha
ano,,,,
2000,83249,0,11,0
2001,179510,0,1492,0
2002,324795,0,1367,0
2003,206036,0,1645,0
2004,41861,0,0,0
2005,124677,0,0,0
2006,107714,0,0,0
2007,175510,0,0,7
2008,0,299707,0,14


In [17]:
# unidade (1º caractere) depois de tirar o '.0'
consulta("""
    select case when ano_arquivo <= 2006 then '2000-2006' else '2007-2026' end as periodo,
           left(regexp_replace(coalesce(nu_idade_n, nu_idade), '\\.0$', ''), 1) as unidade, count(*) as notificacoes
    from ne where coalesce(nu_idade_n, nu_idade) ~ '^([A-Z]|\\d)\\d{3}(\\.0)?$'
    group by 1, 2 order by 1, 3 desc""")

,periodo,unidade,notificacoes
0,2000-2006,A,1050089
1,2000-2006,M,8905
2,2000-2006,I,4871
3,2000-2006,D,3968
4,2000-2006,2,2
5,2000-2006,4,2
6,2000-2006,8,2
7,2000-2006,0,1
8,2000-2006,6,1
9,2000-2006,7,1


In [18]:
# idades em anos impossíveis e conferência com o ano de nascimento (2007+)
consulta("""
    with i as (select nu_ano, ano_nasc, regexp_replace(coalesce(nu_idade_n, nu_idade), '\\.0$', '') as cod from ne)
    select count(*) filter (where cod ~ '^[4A]\\d{3}$' and substr(cod, 2)::int > 110) as anos_acima_de_110,
           count(*) filter (where cod ~ '^4\\d{3}$' and ano_nasc ~ '^\\d{4}$' and nu_ano ~ '^\\d{4}$') as com_ano_nascimento,
           count(*) filter (where cod ~ '^4\\d{3}$' and ano_nasc ~ '^\\d{4}$' and nu_ano ~ '^\\d{4}$'
                            and abs((nu_ano::int - ano_nasc::int) - substr(cod, 2)::int) > 1) as divergem_mais_de_1_ano
    from i""")

,anos_acima_de_110,com_ano_nascimento,divergem_mais_de_1_ano
0,1017,1693287,393


### 8.4 Município de notificação

In [19]:
consulta("""
    select count(*) filter (where id_municip is null) as vazio,
           count(*) filter (where m.cod_mun6 is not null) as bate_com_ibge,
           count(*) filter (where id_municip is not null and m.cod_mun6 is null) as nao_bate
    from ne left join municipio m on m.cod_mun6 = left(ne.id_municip, 6)""")

,vazio,bate_com_ibge,nao_bate
0,0,5598575,2


### 8.5 Classificação: códigos fora do seu período
Os códigos 1 a 4 (confirmado) são da ficha até 2013; 10 a 12, da ficha desde 2014. Aparecem códigos antigos depois de 2014?

In [20]:
consulta("""
    select ano_arquivo as ano, coalesce(coalesce(classi_fin, con_classi), 'vazio') as codigo, count(*) as notificacoes
    from ne where ano_arquivo >= 2014 and coalesce(classi_fin, con_classi, 'vazio') not in ('5', '8', '10', '11', '12')
    group by 1, 2 order by 1, 3 desc""")

,ano,codigo,notificacoes
0,2014,vazio,524
1,2015,vazio,139
2,2016,1,4664
3,2016,vazio,348
4,2016,2,13
5,2016,3,3
6,2016,4,2
7,2017,vazio,179
8,2018,vazio,198
9,2019,vazio,754


### 8.6 Critério e evolução
Não entram na definição de caso, mas descrevem a qualidade da investigação.

In [21]:
consulta("""
    select ano_arquivo as ano,
           round(100.0 * count(criterio) / count(*), 1) as pct_criterio_preenchido,
           round(100.0 * count(evolucao) / count(*), 1) as pct_evolucao_preenchida
    from ne group by 1 order by 1""").set_index("ano").T

ano,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
pct_criterio_preenchido,73.5,68.7,73.1,78.5,84.7,86.5,89.1,82.5,79.0,75.2,...,80.8,77.8,73.5,69.8,82.6,75.0,75.5,75.8,72.4,68.1
pct_evolucao_preenchida,87.2,82.1,82.3,81.4,85.7,85.2,88.5,53.2,48.4,46.7,...,75.1,66.4,69.3,64.6,76.8,71.3,69.8,68.8,66.9,61.1


## 9. Resumo

**Em poucas palavras:** antes de criar a tabela limpa, conferimos coluna por coluna se o bruto tem "sujeira"
(valores que não fazem sentido). Os dados estão bons no geral, com erros pequenos e um grande:

1. **Idade:** em alguns anos (2008, 2012, 2016 a 2019) o arquivo escreveu a idade com `.0` no final (`4023.0`).
   Se a limpeza não tratar isso, perde a idade de quase 1 em cada 3 notificações.
2. **Datas:** poucas datas de sintomas impossíveis (ano 0006, ano 2043), erros de digitação. Ficam vazias.
3. **Sexo:** quase tudo M ou F; uns poucos "ignorado" ficam vazios.
4. **Classificação:** em 2016 alguns casos usam os códigos antigos de "confirmado"; a regra aceita os dois.
5. **Possíveis duplicadas** (seção 6) e **anos sem descartados** (seção 7): tratados pela definição de caso provável
   e pela marca `possivel_duplicata`.

Os detalhes e números estão abaixo.


**Ligação com o IBGE: funciona.**
- 5.598.577 notificações de residentes do Nordeste (2000 a 2026).
- 99,997% têm código que existe no IBGE pelos 6 primeiros dígitos; nos códigos de 7 dígitos, o 7º dígito sempre bate.
- Os 1794 municípios do Nordeste têm notificações.
- Só 167 não batem: quase todos são "estado conhecido, município ignorado" (ex.: `2900000`, `290000`), mais 2 erros de digitação.

**Recorte por residência: capta também quem foi notificado fora.**
- Residentes do Nordeste notificados em outras regiões: de 0,03% (anos antigos) a 1,09% (2025) por ano.
- No Brasil, ~3.700 notificações (0,01%) não têm município de residência e não entram em recorte nenhum.

**Ano da notificação × ano do arquivo: quase sempre igual.**
- Diferenças em poucos anos, até 1.088 registros (2025, com notificações de 2024 e 2026). Há 2 valores sem sentido em 2008 (`20¸`, `2408`).

**Estado de residência: coerente.** 99,99% dos `sg_uf` batem com o município; 90 diferentes e 85 vazios.

**Possíveis duplicadas: 0,7% a 3,3% por ano** têm os mesmos dados de notificação, pessoa e residência. Sem nome ou
documento (dados anonimizados) não dá para ter certeza; o campo de duplicidade do SINAN (`nduplic_n`) está quase
sempre vazio. Na limpeza elas são **marcadas, não apagadas** (`possivel_duplicata`), para medir o efeito.

**Classificação final: o ponto mais importante.**
- **Em 2014, 2015 e de 2022 a 2026 não há nenhuma notificação descartada (código 5).** Nos outros anos, os descartados
  são de 16% a 47%. Ou seja, o arquivo desses anos já vem sem os descartados, e o total de notificações **não é
  comparável** entre os anos.
- Por isso, a série precisa usar a mesma definição em todos os anos: **excluir os descartados** (código 5) em todos.
- Nos anos antigos há classificação vazia (9% a 18% de 2000 a 2006) e um código `0` (até 10% de 2000 a 2003) que não
  está no dicionário: precisam de tratamento.
- Inconclusivos (código 8) vão de 0,4% a 35% conforme o ano.

**Validação coluna por coluna (seção 8)**
- **Datas:** quase todas válidas, mas há valores impossíveis nos primeiros sintomas (de 0006 a 2043), 3 em formato
  inválido, 13 com sintomas *depois* da notificação e 13.016 com sintomas mais de 1 ano antes. Mediana: 3 dias entre
  sintomas e notificação.
- **Sexo:** 6.648 `I` (ignorado), 120 vazios e 1 `O`.
- **Idade: em 2008, 2012 e de 2016 a 2019 a idade vem como decimal (`4023.0`)**. Sem tratar isso, 1,66 milhão de
  idades seriam perdidas. Fora isso: ficha antiga com letras (`A`, `M`, `D`, `I999` = ignorado), ~400 valores sem
  unidade e 1.017 acima de 110 anos. Onde há ano de nascimento, a idade confere em 99,98% dos casos.
- **Município de notificação:** só 2 sem par no IBGE.
- **Classificação:** em 2016 aparecem 4.682 códigos da ficha antiga (1 a 4, confirmados); em 2026 há 11.963 vazios
  (ano em andamento, casos ainda em investigação).
- **Critério e evolução:** preenchidos em 68% a 89% e 47% a 89% das notificações, conforme o ano.

**Regras propostas para a limpeza**
| Coluna | Regra |
|---|---|
| datas | converter para data; inválidas viram nulo |
| `dt_sintomas` | nulo se depois da notificação ou mais de 1 ano antes dela (a notificação continua) |
| `sexo` | M ou F; `I`, `O` e vazio viram nulo |
| `idade_anos` | tirar o `.0`; decodificar as duas fichas (anos = valor; meses, dias e horas = 0); `I999`, sem unidade e acima de 110 viram nulo |
| `cod_mun7_notificacao` | código do IBGE; os 2 sem par viram nulo |
| `caso_confirmado` | códigos 1–4 **em qualquer ano** (inclui os de 2016) e 10–12 |
| `caso_provavel` | tudo menos o 5 (inclui vazios e o código 0) |
